# 06. Sweep で改善し、比較して、後片付けする

**対応するテキスト**: [docs/08_Sweepで改善する.md](../docs/08_Sweepで改善する.md) ／ [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md)

> ## ⚠⚠ このノートブックは **必ず最後まで実行してください**
>
> **5 節の後片付けを実施しないと、ハンズオン終了後も課金が続きます。**
> また **3 節で結果を手元に保存する前にリソースを削除すると、復旧できません。**

実行順序:

1. Sweep Job でハイパーパラメーターを探索する
2. 最良の試行を確認する
3. **すべての実験を 1 枚の表にまとめる**
4. コストを確認する
5. **⚠ 後片付け（必須）**

> [!WARNING]
> **このノートブックの Azure ジョブは Azure 上で実行検証していません。**
> 探索の結果として得られる数値は記載していません。**あなたの環境で確認してください。**

> ⚠ **サブスクリプション ID を書き込んだノートブックをコミットしないでください。**

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください（01・03〜05 と同じ値）
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "il-pickplace-env@latest"
DATA_ASSET_NAME = "il-pickplace-demos"
EXPERIMENT_SWEEP = "il-sweep"

#  これまでに使ったすべての実験名（3 節でまとめて集計します）
ALL_EXPERIMENTS = ["il-setup-check", "il-demos", "il-bc", "il-dagger-gail", "il-sweep"]

#  ⚠ クラスターの max_instances と vCPU クォータを超えないこと
MAX_CONCURRENT = 3

TAGS = {
    "project": "il-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow

ml_client = MLClient(
    credential=DefaultAzureCredential(exclude_interactive_browser_credential=False),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

tracking_uri = getattr(ws, "mlflow_tracking_uri", None)
if tracking_uri is None:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
print("MLflow 追跡先を設定しました。")

## 1. Sweep Job — 「学習量」を探索する

[04_bc_job.ipynb](04_bc_job.ipynb) で分かったのは、
**同じ更新回数でも「データ量と反復回数のバランス」で成績が変わる**ということでした。

そこで探索するのも **学習量を決める 2 つの値**にします。

| 探索する引数 | 候補 | 意味 |
|---|---|---|
| `--epochs` | 37 / 150 / 600 | データを何周するか |
| `--batch-size` | 16 / 32 / 64 | 1 回の更新に使う件数 |

デモ数は **04 で最良だった 200 件に固定**します。

### ⚠ デモ収集を Sweep の中に入れない理由

**Sweep の各試行は、毎回ゼロからやり直します。**
デモ収集を試行の中に入れると、**試行ごとに `scores.json` が変わり、`normalized_return` の物差しが揃わなくなります。**
だから [03](03_collect_demos_job.ipynb) で**データ資産として 1 度だけ作り、全試行で共有**します。

> 出典（Microsoft 公式）: [Hyperparameter tuning a model (v2)](https://learn.microsoft.com/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2)

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.sweep import Choice

TRIAL_COMMAND = (
    "python train_il.py"
    " --algo bc"
    " --demos-dir ${{inputs.demos}}"
    " --output-dir ${{outputs.model}}"
    " --n-demo-episodes ${{inputs.n_demo_episodes}}"
    " --epochs ${{inputs.epochs}}"
    " --batch-size ${{inputs.batch_size}}"
    " --seed ${{inputs.seed}}"
)

trial_job = command(
    code="../src",
    command=TRIAL_COMMAND,
    inputs=dict(
        demos=Input(type=AssetTypes.URI_FOLDER, path=f"azureml:{DATA_ASSET_NAME}@latest"),
        n_demo_episodes=200,   # 固定（04 で最良だった条件）
        epochs=150,            # ← 既定値。下の sweep で上書きされる
        batch_size=32,         # ← 同上
        seed=0,                # 固定
    ),
    outputs=dict(model=Output(type=AssetTypes.URI_FOLDER)),
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    tags={**TAGS, "algo": "bc", "phase": "sweep"},
)
print(TRIAL_COMMAND)

### 主要メトリックを `normalized_return` にする理由

**「検証損失が一番小さい設定」を選んではいけません。**

BC の検証損失は「専門家の行動をどれだけ正確に当てられたか」ですが、
**私たちが知りたいのは「ロボットが実際に物体を運べるか」**です。この 2 つは一致しません。

> "the variability based on the stopping criteria **due to the different objectives in training and evaluation**"
>
> 出典（参考情報・学術論文）: Mandlekar et al., *What Matters in Learning from Offline Human Demonstrations for Robot Manipulation*, arXiv:2108.03298

`normalized_return` は **環境で 20 エピソード動かして測った成績**を、
「一様ランダム = 0 / 専門家 = 1」に直した値です（[../src/train_il.py](../src/train_il.py)）。

> ⚠ **`primary_metric` は、学習スクリプトが記録するメトリック名と完全一致していなければなりません。**

In [ ]:
sweep_job = trial_job(
    epochs=Choice(values=[37, 150, 600]),
    batch_size=Choice(values=[16, 32, 64]),
).sweep(
    compute=COMPUTE_NAME,
    sampling_algorithm="grid",              # 3 × 3 = 9 通りを全部試す
    primary_metric="normalized_return",     # ← train_il.py が記録する名前と完全一致
    goal="Maximize",
    max_total_trials=9,                     # ← コストの上限
    max_concurrent_trials=MAX_CONCURRENT,
    timeout=6 * 60 * 60,                    # ← 全体の時間上限（秒）
)
sweep_job.experiment_name = EXPERIMENT_SWEEP
sweep_job.display_name = "sweep_bc_epochs_x_batchsize"

returned_sweep = ml_client.jobs.create_or_update(sweep_job)
print("Sweep:", returned_sweep.name)
print("studio:", returned_sweep.studio_url)

### ⚠ 早期終了ポリシーについて（本 Sweep では付けていません）

> **早期終了ポリシーは「主要メトリックがログに記録されるたびに 1 区間」と数えます。**
>
> 出典（Microsoft 公式）: [Hyperparameter tuning a model (v2) — Early termination](https://learn.microsoft.com/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2#early-termination)

`normalized_return` は **学習の最後に 1 回だけ**記録されるため、区間が 1 つしかなく、打ち切る余地がありません。

**効かせたい場合は、主要メトリックを `epoch_eval_success_rate` に変えます。**
[../src/train_il.py](../src/train_il.py) は BC の各エポック終了時に、**step 付きで**この値を記録しています。

```python
from azure.ai.ml.sweep import BanditPolicy

sweep_job = trial_job(
    epochs=Choice(values=[150, 600]),
    batch_size=Choice(values=[16, 32, 64]),
).sweep(
    compute=COMPUTE_NAME,
    sampling_algorithm="grid",
    primary_metric="epoch_eval_success_rate",   # ← step 付きで記録されるメトリック
    goal="Maximize",
    max_total_trials=6,
    max_concurrent_trials=MAX_CONCURRENT,
    early_termination=BanditPolicy(
        slack_factor=0.2, evaluation_interval=1, delay_evaluation=5
    ),
)
```

> ⚠ **`delay_evaluation` を必ず入れてください。** 学習の序盤は成績が安定しません。

> ⚠ **エポックごとの評価は無料ではありません。**
> 1 回の評価で **20 エピソード × 50 ステップ = 1,000 ステップ**の物理シミュレーションが走ります。
> **エポック数が多い設定では、評価のほうが学習より重くなることがあります。**

In [ ]:
ml_client.jobs.stream(returned_sweep.name)

sweep_status = ml_client.jobs.get(returned_sweep.name)
print("ステータス:", sweep_status.status)

## 2. 最良の試行を確認する

Sweep の子ジョブ（trial）は**親ジョブと同じ実験に記録されます**。

**studio の［平行座標プロット］も必ず見てください。** どの値が効いているかが一目で分かります。

In [ ]:
import pandas as pd

sweep_runs = mlflow.search_runs(experiment_names=[EXPERIMENT_SWEEP], output_format="pandas")

COLS = {
    "tags.mlflow.runName": "run_name",
    "params.epochs": "epochs",
    "params.batch_size": "batch_size",
    "params.n_demo_episodes": "n_demos",
    "params.seed": "seed",
    "metrics.n_transitions": "n_transitions",
    "metrics.eval_success_rate": "success_rate",
    "metrics.eval_return_mean": "return_mean",
    "metrics.normalized_return": "normalized",
}
available = {k: v for k, v in COLS.items() if k in sweep_runs.columns}
sw = sweep_runs[list(available)].rename(columns=available).dropna(subset=["normalized"])
for col in ("epochs", "batch_size", "n_demos", "seed", "n_transitions"):
    if col in sw.columns:
        sw[col] = pd.to_numeric(sw[col], errors="coerce")

#  実際の勾配更新回数を計算して並べる（何が効いたのかを見るため）
if {"n_transitions", "batch_size", "epochs"}.issubset(sw.columns):
    sw["n_updates"] = (sw["n_transitions"] // sw["batch_size"] * sw["epochs"]).astype(int)

sw = sw.sort_values("normalized", ascending=False)
print("試行数:", len(sw))
if len(sw):
    best = sw.iloc[0]
    print(f"最良: epochs={best.get('epochs')} batch_size={best.get('batch_size')} "
          f"normalized_return={best['normalized']:.3f} success_rate={best.get('success_rate')}")
sw

In [ ]:
import matplotlib.pyplot as plt

if {"n_updates", "normalized"}.issubset(sw.columns) and len(sw):
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for bs, sub in sw.groupby("batch_size"):
        sub = sub.sort_values("n_updates")
        ax.plot(sub["n_updates"], sub["normalized"], marker="o", label=f"batch_size={int(bs)}")
    ax.set_xscale("log")
    ax.set_xlabel("勾配更新の回数（対数軸）")
    ax.set_ylabel("normalized_return")
    ax.set_title("学習量と正規化リターン（デモ 200 件・seed 0）")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("データが足りません。Sweep の完了を待ってください。")

### 結果の読み方

- **`epochs` を増やすと成績が上がり、どこかで頭打ちになる**はずです。
  頭打ちになったら、**それ以上の学習量は費用の無駄**です。
- **`batch_size` を小さくすると、同じ `epochs` でも更新回数が増えます。** 2 つの軸は独立ではありません。
- **1 位の設定だけを見て決めないでください。**

> [!IMPORTANT]
> **この Sweep は `seed` を固定しています。**
> [04](04_bc_job.ipynb) で見たとおり、**シードを変えるだけで成功率が大きく動きます。**
> **上位が僅差なら「差は無い」と判断するのが正しい**です。
> 優劣を判定したい場合は、**上位の設定だけを 3 シードで再実行**してください。

## 3. すべての実験を 1 枚の表にまとめる

**これがハンズオンの成果物になります。**

> ⚠ `search_runs` が返すメトリックは **各メトリックの最後の値**です。
> 学習曲線が必要な場合は `MlflowClient().get_metric_history(run_id, key)` を使ってください。
>
> 出典（Microsoft 公式）: [Query & compare experiments and runs with MLflow](https://learn.microsoft.com/azure/machine-learning/how-to-track-experiments-mlflow?view=azureml-api-2)

In [ ]:
existing = [name for name in ALL_EXPERIMENTS if mlflow.get_experiment_by_name(name) is not None]
for name in ALL_EXPERIMENTS:
    if name not in existing:
        print(f"[INFO] 実験 '{name}' は見つかりませんでした（未実行かもしれません）")

if not existing:
    raise RuntimeError("実験が 1 つも見つかりません。01・03 から順に実行してください。")

all_runs = mlflow.search_runs(experiment_names=existing, output_format="pandas")

ALL_COLS = {
    "tags.mlflow.runName": "run_name",
    "tags.algo": "tag_algo",
    "params.algo": "algo",
    "params.n_demo_episodes": "n_demos",
    "params.epochs": "epochs",
    "params.batch_size": "batch_size",
    "params.dagger_timesteps": "dagger_steps",
    "params.gail_timesteps": "gail_steps",
    "params.seed": "seed",
    "metrics.n_transitions": "n_transitions",
    "metrics.eval_success_rate": "success_rate",
    "metrics.eval_return_mean": "return_mean",
    "metrics.eval_return_std": "return_std",
    "metrics.normalized_return": "normalized",
    "metrics.random_mean": "random_mean",
    "metrics.expert_mean": "expert_mean",
    "metrics.expert_success_rate": "expert_success_rate",
    "run_id": "run_id",
}
available = {k: v for k, v in ALL_COLS.items() if k in all_runs.columns}
summary = all_runs[list(available)].rename(columns=available)

if "normalized" in summary.columns:
    summary = summary.sort_values("normalized", ascending=False, na_position="last")

summary.to_csv("all_runs_comparison.csv", index=False, encoding="utf-8-sig")
print(f"\n取得した Run 数: {len(summary)} → all_runs_comparison.csv に保存しました")
summary.head(30)

In [ ]:
#  手法ごとの要約（報告に使う数値）
if {"algo", "success_rate"}.issubset(summary.columns):
    report = (
        summary.dropna(subset=["success_rate"])
        .groupby("algo")[["success_rate", "return_mean", "normalized"]]
        .agg(["count", "mean", "min", "max"])
    )
    print("=== 手法ごとの要約 ===")
    print(report)
    print()
    print("⚠ count が 1 の行は『1 回しか測っていない』という意味です。比較の根拠にはできません。")
    print("⚠ 学習量（勾配更新回数・環境ステップ数）が手法ごとに違うことにも注意してください。")

## 4. コストを確認する

> **本ハンズオンは具体的な金額を記載しません。** 価格はリージョン・VM サイズ・時期で変わり、
> **本ハンズオンは Azure 上で実行検証していない**ためです。**必ず以下で実際の値を確認してください。**

| 確認するもの | 場所 |
|---|---|
| **実際にかかった費用** | Azure Portal →［コスト管理］→［コスト分析］→ **タグ `project = il-workshop` でフィルター** |
| VM の時間単価 | [Azure Machine Learning 価格](https://azure.microsoft.com/pricing/details/machine-learning/) |
| 見積もり | [Azure 料金計算ツール](https://azure.microsoft.com/pricing/calculator/) |

> ⚠ **本ハンズオンは物理シミュレーションを伴うため、CartPole のような題材より 1 ジョブが重くなります。**
> ローカルでの参考値は [docs/09 の 9.3](../docs/09_評価・コスト・後片付け.md) にあります。

In [ ]:
cluster = ml_client.compute.get(COMPUTE_NAME)
print("=== コンピューティング クラスター ===")
print(f"  名前                       : {cluster.name}")
print(f"  VM サイズ                  : {cluster.size}")
print(f"  min_instances              : {cluster.min_instances}  ← 0 であること")
print(f"  max_instances              : {cluster.max_instances}")
print(f"  idle_time_before_scale_down: {cluster.idle_time_before_scale_down} 秒")

print("\n=== すべてのコンピューティング（種別と状態） ===")
for c in ml_client.compute.list():
    kind = str(getattr(c, "type", "")).lower()
    state = str(getattr(c, "state", "")) or "-"
    is_instance = "instance" in kind or type(c).__name__ == "ComputeInstance"
    mark = "  ⚠ 停止してください" if (is_instance and state.lower() == "running") else ""
    print(f"  {c.name:28s} type={kind:18s} state={state}{mark}")

## 5. ⚠ 後片付け（必須）

> ⚠⚠ **リソースを削除すると、MLflow の記録も成果物も復旧できません。**
> **`all_runs_comparison.csv` を手元に保存してから**実行してください。

### 5-1. 実行中のジョブを確認する

In [ ]:
ACTIVE = ("Running", "Queued", "Preparing", "Starting", "NotStarted")

active_jobs = [job for job in ml_client.jobs.list() if str(job.status) in ACTIVE]
for job in active_jobs:
    print(f"⚠ 実行中: {job.name}  status={job.status}  display_name={job.display_name}")

if not active_jobs:
    print("実行中のジョブはありません。")

In [ ]:
# ⚠ 実行すると、上で列挙されたジョブをすべてキャンセルします。
#    必要なジョブが走っていないか確認してから、コメントを外してください。
#
# for job in active_jobs:
#     ml_client.jobs.begin_cancel(job.name)
#     print("キャンセル要求:", job.name)

### 5-2. コンピューティング クラスターを削除する

**`min_instances=0` ならノードの課金は止まりますが、クラスターの定義自体は残ります。**

In [ ]:
# ⚠ 実行するとクラスターを削除します。コメントを外してください。
#
# ml_client.compute.begin_delete(COMPUTE_NAME).wait()
# print("削除しました:", COMPUTE_NAME)

print("上のコメントを外して実行するか、Azure ML studio の［コンピューティング］から削除してください。")

### 5-3. すべて不要な場合 — リソース グループごと削除する（最も確実）

> ⚠⚠ **元に戻せません。3 節の CSV を保存してから実行してください。**

```powershell
az group delete --name <リソース グループ名> --yes --no-wait
```

**`min_instances=0` にしていても、ストレージ アカウント・Key Vault・Container Registry など
ワークスペース付属リソースの課金は続きます。**

### 5-4. ローカルの後片付け

```powershell
# conda 環境を削除する
conda env remove -n il-panda
```

ノートブックを実行したフォルダーに **`mlflow.db`** が残っていれば削除して構いません
（`.gitignore` で除外済みです）。

### 5-5. 残す場合の判断

| 判断 | 残すもの | 消すもの |
|---|---|---|
| 継続検証する | ワークスペース、環境、データ資産、MLflow の記録 | **コンピューティング クラスター**（同じ手順で作り直せます） |
| 記録だけ残す | ワークスペース（記録の保管庫） | コンピューティング一式 |
| すべて終了 | 手元に保存した CSV のみ | **リソース グループごと** |

## ✅ 最終チェックリスト

- [ ] Sweep Job を実行し、**`max_total_trials` と `timeout` を設定した**（コスト上限）
- [ ] studio の**平行座標プロット**を確認した
- [ ] **`all_runs_comparison.csv` を手元に保存した**
- [ ] **Microsoft Cost Management で実際の費用を確認した**
- [ ] **実行中のジョブが残っていないことを確認した**
- [ ] **コンピューティング クラスターを削除した**（または継続方針を決めた）
- [ ] ローカルの `il-panda` 環境と `mlflow.db` を整理した
- [ ] **サブスクリプション ID を書いたノートブックをコミットしていない**

---

**お疲れさまでした。** 結果のまとめ方は [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md) を参照してください。